# Serve AI4Bharat IndicTrans2 on Colab GPU (T4) -- NOT ADOPTED, see status below

**Status: attempted, blocked on a KV-cache incompatibility, not adopted.**
`render/translate.py` uses `--backend libretranslate` instead (see its module
docstring). This notebook is kept as a diagnostic record in case current
`transformers` ever reworks its Cache API in a way IndicTrans2's remote code
becomes compatible with again -- do not expect it to produce a working server
as-is.

Why this was attempted: `render/translate.py`'s Gemini backend is capped at
the Google AI Studio free tier's 20-requests/day/model quota -- confirmed by
hitting `RESOURCE_EXHAUSTED` on two different model names in the same day
while translating this project's ~2,591-notification corpus. IndicTrans2 is
self-hosted (no daily call ceiling) and purpose-built for exactly the 22
Eighth Schedule languages in `render/translate.py`'s `INDIAN_LANGUAGES` --
the right choice on paper.

**Five real compatibility bugs between IndicTrans2's ~2023 remote code and
current `transformers` were found and fixed live** (all baked into the
`server.py` written below, which does get as far as loading the model):
1. `PreTrainedTokenizerBase` moved out of `transformers.tokenization_utils`.
2. `transformers.onnx` was removed entirely (needed a no-op stub module).
3. `IndicTransTokenizer.__init__` sets `self.unk_token = ...` before calling
   `super().__init__()`, which crashes current `transformers`' `__setattr__`
   (it expects `_special_tokens_map` to already exist) -- fixed by patching
   `PreTrainedTokenizerBase.__setattr__` to lazily initialize it.
4. `tie_weights()` is called with a `recompute_mapping=` kwarg (and, at a
   second call site, `missing_keys=` too) that IndicTrans2's overridden
   `tie_weights()` doesn't accept -- fixed by patching the dynamically
   loaded class's `tie_weights` to accept and discard extra kwargs.
5. `_tie_or_clone_weights` was removed from `PreTrainedModel` entirely --
   re-added its historic implementation as a monkeypatch.

**The 6th bug is where this stopped.** `modeling_indictrans.py`'s decoder
assumes the old tuple-of-tuples KV-cache format throughout (e.g.
`past_key_values[idx]`), but current `transformers` passes an
`EncoderDecoderCache` object -- a much more complex structure (built for
sliding-window/hybrid caching, confirmed via `dir()`: no `to_legacy_cache()`,
only low-level methods like `update_indexer`/`update_conv_state`) that isn't
subscriptable and has no documented conversion back to the legacy tuple
shape. Disabling caching (`use_cache=False`) avoids the crash but breaks
*correctness*, not just speed -- the model's custom decoder computes
position embeddings from the cache length, and without one it produced
garbled, non-repeating-but-nonsensical output rather than a translation.

**Pinning an old, actually-matching `transformers` version is also a dead
end on this Colab image**, confirmed live: `transformers==4.33.2` (the
version AI4Bharat's own `install.sh` names as its floor) needs
`tokenizers<0.14`, whose Rust/PyO3 bindings predate Python 3.13's C API --
even installing a fresh Rust toolchain via `rustup` (not just apt's stale
one) and letting it build from source for ~10 minutes ends in `error:
subprocess-exited-with-error` building the `tokenizers` wheel. This isn't a
quick-fix version window; it would need patching PyO3 itself.


In [ ]:
!nvidia-smi -L

In [ ]:
!pip install -q transformers sentencepiece flask
!pip install -q git+https://github.com/VarunGumma/IndicTransToolkit.git

In [ ]:
%%writefile /content/server.py
import os, sys, types

import transformers.tokenization_utils as _tu
if not hasattr(_tu, "PreTrainedTokenizerBase"):
    from transformers.tokenization_utils_base import PreTrainedTokenizerBase as _PTB
    _tu.PreTrainedTokenizerBase = _PTB

_onnx = types.ModuleType("transformers.onnx")
class OnnxConfig: pass
class OnnxSeq2SeqConfigWithPast(OnnxConfig): pass
_onnx.OnnxConfig = OnnxConfig
_onnx.OnnxSeq2SeqConfigWithPast = OnnxSeq2SeqConfigWithPast
_onnx_utils = types.ModuleType("transformers.onnx.utils")
_onnx_utils.compute_effective_axis_dimension = lambda *a, **k: 0
_onnx.utils = _onnx_utils
sys.modules["transformers.onnx"] = _onnx
sys.modules["transformers.onnx.utils"] = _onnx_utils

import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from IndicTransToolkit.processor import IndicProcessor
from flask import Flask, request, jsonify

CKPT = "ai4bharat/indictrans2-en-indic-dist-200M"
device = "cuda" if torch.cuda.is_available() else "cpu"

print("Loading tokenizer/model...", flush=True)
tokenizer = AutoTokenizer.from_pretrained(CKPT, trust_remote_code=True)
model = AutoModelForSeq2SeqLM.from_pretrained(CKPT, trust_remote_code=True).to(device).eval()
ip = IndicProcessor(inference=True)
print("Model ready.", flush=True)

def translate(text: str, tgt_lang: str, src_lang: str = "eng_Latn") -> str:
    batch = ip.preprocess_batch([text], src_lang=src_lang, tgt_lang=tgt_lang)
    inputs = tokenizer(batch, truncation=True, padding="longest", return_tensors="pt").to(device)
    with torch.no_grad():
        generated = model.generate(**inputs, use_cache=True, min_length=0, max_length=256, num_beams=5)
    with tokenizer.as_target_tokenizer():
        decoded = tokenizer.batch_decode(generated.detach().cpu().tolist(), skip_special_tokens=True)
    return ip.postprocess_batch(decoded, lang=tgt_lang)[0]

app = Flask(__name__)

@app.route("/translate", methods=["POST"])
def _translate_route():
    data = request.get_json(force=True)
    text = data.get("text", "")
    tgt_lang = data.get("tgt_lang")
    src_lang = data.get("src_lang", "eng_Latn")
    if not text or not tgt_lang:
        return jsonify({"error": "text and tgt_lang are required"}), 400
    return jsonify({"translation": translate(text, tgt_lang, src_lang)})

@app.route("/health", methods=["GET"])
def _health():
    return jsonify({"status": "ready", "model": CKPT, "device": device})

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5000)

In [ ]:
import os, subprocess, time, urllib.request
from google.colab import userdata

env = os.environ.copy()
env["HF_TOKEN"] = userdata.get("HF_TOKEN")

server_log = open("/content/server.log", "w")
server = subprocess.Popen(["python3", "/content/server.py"], env=env, stdout=server_log, stderr=subprocess.STDOUT)

print("Waiting for model to load (this downloads the checkpoint the first time)...")
ready = False
for _ in range(180):
    time.sleep(2)
    try:
        with urllib.request.urlopen("http://localhost:5000/health", timeout=3) as resp:
            if resp.status == 200:
                ready = True
                break
    except Exception:
        pass
    if server.poll() is not None:
        break

if ready:
    print("Server ready on :5000")
else:
    print("Server did not become ready -- check /content/server.log:")
    print(open("/content/server.log").read()[-4000:])

In [ ]:
# Expose via Cloudflare quick tunnel, same as serve_mistral_colab.ipynb.
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x cloudflared
import subprocess, re, time
tun = subprocess.Popen(['./cloudflared', 'tunnel', '--url', 'http://localhost:5000'],
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
t0 = time.time()
while time.time() - t0 < 30:
    line = tun.stdout.readline()
    m = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if m:
        url = m.group(0)
        break
print('INDICTRANS2_COLAB_URL=' + (url or 'NOT FOUND -- re-run this cell'))

**This notebook is not wired into `render/translate.py`'s CLI** (no
`--backend indictrans2` exists) since it was never gotten past the KV-cache
bug above. If a future `transformers`/`IndicTransToolkit` release fixes the
cache-object compatibility, `render/translate.py` documents the same
`translate_record_<backend>()` extension point used for the LibreTranslate
backend actually in use -- add `translate_record_indictrans2()` there,
reusing the `server.py` written by this notebook (cells above still work up
to model loading) once the cache issue is resolved.